# 生成結果を 1 文字ずつ表示する (ストリーミング)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm/llm_streaming_local.ipynb)

NLP クックブック 13-02 ／ 大規模言語モデル (LLM)

生成が終わるまで待たずに、出力されたそばから表示したい。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).eval()

messages = [{"role": "user", "content": "自然言語処理とは何か、3 文で説明してください。"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)   # 標準出力に逐次表示
_ = model.generate(**inputs, max_new_tokens=120, do_sample=False, streamer=streamer, pad_token_id=tokenizer.eos_token_id)

自然言語処理（NLP）とは、自然言語をコンピュータが理解、生成、分析できるようにする技術分野です。これはテキストデータから意味や感情を抽出し、さまざまなタスク（例：文章の分類、機械翻訳、言語理解など）に活用することを目的としています。このプロセスでは、機械学習や人工知能の技術が主に用いられ、人間の言語をコンピュータが処理しやすい形式に変換することを目指します。


In [2]:
# 自分のコードで受け取りたい場合 (Web アプリに流すなど): 別スレッドで生成し、イテレータから読む
from threading import Thread
from transformers import TextIteratorStreamer

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
thread = Thread(target=model.generate, kwargs=dict(**inputs, max_new_tokens=60, do_sample=False, streamer=streamer, pad_token_id=tokenizer.eos_token_id))
thread.start()
chunks = [chunk for chunk in streamer]
thread.join()
print(len(chunks), "個の断片:", chunks[:8])

61 個の断片: ['自然', '言語', '処理', '', '', '', '', '']
